# Decision Models, Optimization, and Feedback

Turn predictions into constrained decisions, evaluate a contextual policy offline, and measure consequences before rollout.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Practice case · Who gets the next maintenance slot?

HarborLink has a queue and a finite number of technician slots. Leena owns the decision policy. She needs to translate delay-risk estimates into a plan and explain why some tickets are deferred for review.

The workplace scenario is fictional.


## Design the decision layer


### Lecture 01 · Calibrated predictions and decisions under cost

Derive an action threshold from costs and measure the option to defer.

#### Give the uncertainty somewhere to go

A maintenance alert is neither clearly routine nor obviously urgent. The model is moderately confident and a short review is available.

Write the action and review costs before choosing a threshold. Track how often the policy defers and whether the team can handle that load.

#### How it works

A predictive model estimates outcomes. A policy chooses an action given the estimate, constraints and consequences. For binary action with two wrong-decision costs and zero cost for correct decisions, act when the probability exceeds the false-positive cost divided by the sum of error costs. Deferral adds a third option with its own cost and capacity. A reliability diagram checks calibration across groups of forecasts.

##### Put utility above the score
Write a table of actions and their costs conditional on each outcome. Take expectations using the estimated probability, then compare these expected costs with a review option. On historical cases, measure realized outcomes and costs separately. Compare calibration by slice because a score can be calibrated overall and biased in a smaller group. If the options consume shared reviewer time, defer thresholds alone cannot enforce that budget.

#### Worked example

If unnecessary dispatch costs 2 and a missed issue costs 8, the two-action threshold is 0.2. A low-cost review step can be preferable to either action for some scores.

#### Try it

Change the review cost and compare decisions, realized losses, and calibration on the same cases.


In [ ]:
from collections import Counter


def choose_action(probability, review_cost, false_positive=2, false_negative=8):
    costs = {"act": (1 - probability) * false_positive, "wait": probability * false_negative,
             "review": review_cost}
    return min(costs, key=costs.get)


cases = [(0.1, 0), (0.19, 1), (0.4, 0), (0.85, 1)]
for review_cost in (0.5, 1.5, 2.0):
    actions = [choose_action(score, review_cost) for score, _ in cases]
    realized = sum(
        review_cost if action == "review" else 2 * (1 - truth) if action == "act" else 8 * truth
        for action, (_, truth) in zip(actions, cases)
    )
    print(review_cost, Counter(actions), realized)
for low, high in ((0, 0.25), (0.25, 0.75), (0.75, 1.01)):
    bucket = [(score, truth) for score, truth in cases if low <= score < high]
    if bucket:
        print("calibration bucket", (low, high), "count", len(bucket),
              "mean forecast", sum(score for score, _ in bucket) / len(bucket),
              "observed rate", sum(truth for _, truth in bucket) / len(bucket))
print("Brier score", sum((score - truth) ** 2 for score, truth in cases) / len(cases))
assert choose_action(0.19, 2) == "wait"
assert choose_action(0.19, 0.5) == "review"


**Compare your result**

Lower review cost expands the range where review is attractive. Calibration quality and policy value are different measures.


### Lecture 02 · Constrained optimization and planning

Choose a feasible assignment under shared resources and compare it with greedy ranking.

#### Several good jobs compete for one crew

The highest-value job needs both available time blocks. Two smaller jobs fit together and can be finished today.

Solve for the set of jobs rather than the rank of an individual ticket. If a compulsory job cannot fit, escalate the schedule instead of dropping it silently.

#### How it works

When decisions share a constraint, a per-item score cannot choose the full action set. Represent choices as variables with an objective and explicit constraints. For small instances, exhaustive search is usable as a reference. For larger instances, a solver such as CP-SAT can optimize discrete assignments under a time limit. For sequential choices, planning adds transitions and a horizon.

##### Separate feasibility from performance
Start with a tiny instance where you can enumerate every plan and verify the optimum. Then use variables for task inclusion, precedence, and assignment as you extend the model. Evaluate a greedy baseline against the same rules. For a bigger solver, record objective value, bounds, elapsed time, and status. Defining who handles an infeasible instance matters when the plan determines real work.

#### Worked example

One job pays 9 units and occupies both available slots. Two smaller jobs pay 6 units each and fit together. Taking the highest-value single job leaves three units of value on the table.

#### Try it

Compare greedy and exhaustive assignment under two capacity settings. Add a mandatory task and report infeasibility when it cannot fit.


In [ ]:
from itertools import combinations

jobs = (("large", 2, 9), ("small-a", 1, 6), ("small-b", 1, 6))


def assign(jobs, capacity, required=()):
    feasible = (
        (sum(value for _, _, value in subset), tuple(name for name, _, _ in subset))
        for size in range(len(jobs) + 1)
        for subset in combinations(jobs, size)
        if sum(slots for _, slots, _ in subset) <= capacity
        and set(required) <= {name for name, _, _ in subset}
    )
    return max(feasible, default=None)


def greedy(jobs, capacity):
    value = 0
    for _, slots, benefit in sorted(jobs, key=lambda job: -job[2]):
        if slots <= capacity:
            value += benefit
            capacity -= slots
    return value


assert assign(jobs, 2) == (12, ("small-a", "small-b"))
assert assign(jobs, 1, {"large"}) is None
for capacity in (1, 2):
    print("capacity", capacity, "greedy", greedy(jobs, capacity), "optimal", assign(jobs, capacity))


**Compare your result**

Enforce capacity and required tasks before comparing objective values. In production, report a feasible incumbent and solver status when the time limit prevents proving optimality.


## Learn and evaluate from feedback


### Lecture 03 · Contextual bandits and off-policy evaluation

Estimate a target policy from logged actions, rewards and propensities while checking support.

#### History contains the actions somebody chose

The log has thousands of observations, but a rarely selected region has very little evidence for the new policy's proposed action.

Record the logging policy's action probabilities and measure overlap. Examine effective sample size and stop the comparison if the target action has no historical support.

#### How it works

A contextual bandit observes context, chooses one action and receives feedback for that action. Logged outcomes are therefore selected by the behavior policy. Inverse propensity scoring reweights a logged reward by target probability divided by logging probability. For a deterministic target, the numerator is an indicator. A policy that chooses actions with zero logging probability cannot be evaluated this way in those contexts.

##### Check overlap before reporting value
A log provides a reward only for the action it took. A deterministic target puts weight only on matching logged actions, but it requires support for its chosen action in every context being evaluated. Compute effective sample size as the squared sum of weights divided by their sum of squares. A confidence interval or sensitivity analysis is more informative than extra decimal places. If the logger's probabilities were never recorded, an assumed replay estimator may tell you more about the assumptions than the policy.

#### Worked example

When an action has probability one quarter under the logger and the new policy always chooses it, a matching action contributes weight four. High weights can make a few examples dominate the estimate.

#### Try it

Compute the weighted mean and effective sample size, then set one required logging probability to zero.


In [ ]:
def ips(rows):
    # Row fields: logged action, reward, full logging probabilities, target action.
    if not rows or any(probabilities.get(target, 0) <= 0 for _, _, probabilities, target in rows):
        return None
    weights = [1 / probabilities[target] if action == target else 0
               for action, _, probabilities, target in rows]
    estimate = sum(weight * row[1] for weight, row in zip(weights, rows)) / len(rows)
    squares = sum(weight ** 2 for weight in weights)
    effective_size = sum(weights) ** 2 / squares if squares else 0
    return estimate, effective_size


logged = [("A", 1, {"A": 0.25, "B": 0.75}, "A"),
          ("B", 1, {"A": 0.25, "B": 0.75}, "A"),
          ("B", 0, {"A": 0.25, "B": 0.75}, "A"),
          ("B", 0, {"A": 0.25, "B": 0.75}, "A")]
assert ips(logged) == (1.0, 1.0)
assert ips([("B", 1, {"B": 1.0}, "A")]) is None
print("IPS and effective sample size", ips(logged))


**Compare your result**

The estimate must report missing support even if the logged action on that row is different. Doubly robust estimators can also use a reward model. Validate their assumptions before comparing policies.


### Lecture 04 · Sequential decisions and safe policy rollout

Compare myopic choices with finite-horizon planning and set a release contract for feedback-driven policies.

#### Saving minutes now can cost a visit later

A policy defers tricky jobs to improve today's completed jobs per hour. Repeat visits rise over the next reporting period.

Include future workload and unresolved work in the objective. Evaluate a candidate with a shadow workload and connect delayed outcomes to the original policy decision.

#### How it works

A one-step reward can favor actions that make later outcomes worse. A Markov decision process represents state, actions, transition dynamics and reward. Offline reinforcement learning and return-conditioned sequence models such as Decision Transformer learn from trajectories. Prompting for a high return does not provide evidence for actions outside the training distribution. Delayed outcomes, missing feedback, and shifts in the data-collection policy can all distort release decisions.

##### Account for actions that change what happens next
A one-step bandit ignores subsequent state. Planning with a transition model allows a choice now to change later workload and rewards. Offline trajectory methods need coverage for the sequences a policy might produce, and observational logs can omit important influences. For a pilot, freeze a baseline, decide which decisions stay with staff, and set clear guardrails on repeat visits or unresolved requests.

#### Worked example

Deferring an inspection frees a slot immediately but increases the chance of a later repeat visit. A policy that counts only completed tasks per hour learns to defer expensive cases.

#### Try it

Compare immediate reward and two-step return in a small model. Specify what would block a pilot even if mean reward improves.


In [ ]:
# Deterministic transitions for a two-period planning fixture.
transitions = {
    "pending": {"defer": (3, "repeat"), "inspect": (1, "resolved")},
    "repeat": {"revisit": (-5, "resolved")},
    "resolved": {"hold": (0, "resolved")},
}


def plan(state, horizon, discount=1.0):
    if horizon == 0:
        return 0.0, ()
    candidates = []
    for action, (immediate, following) in transitions[state].items():
        future, suffix = plan(following, horizon - 1, discount)
        candidates.append((immediate + discount * future, (action, *suffix)))
    return max(candidates)


assert plan("pending", 1)[1][0] == "defer"
assert plan("pending", 2)[1][0] == "inspect"
print("horizons one and two", plan("pending", 1), plan("pending", 2))


**Compare your result**

State the supported contexts, workload limits, critical outcomes and stop rule. A simulator or an offline estimate can justify a shadow evaluation only within its assumptions.


## Course project

Deliver a decision policy for a synthetic service queue, comparing thresholding, constrained assignment and a feedback policy.

Name the objective and action costs. Provide a defer route, enforce capacity, and report support and effective sample size for offline comparisons.
